# Zelle 1: Initialisierung und Importe
In dieser Zelle werden alle benötigten Bibliotheken geladen und die Pfade für die lokale Ausführung in VS Code konfiguriert.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataframe_manager import DataFrameManager
from stft_processor import calculate_stft

# Pfad zum Datenordner (relativ zum aktuellen Verzeichnis)
data_path = '../data'

if os.path.exists(data_path):
    print("Datenordner gefunden. Initialisierung abgeschlossen.")
else:
    print(f"Warnung: Datenordner nicht gefunden unter: {os.path.abspath(data_path)}")

# Zelle 2: Konfiguration und Daten laden
Hier werden die sensor-spezifischen Parameter definiert und die Signaldaten über den `DataFrameManager` eingelesen.

In [ ]:
# --- INDIVIDUELLE PARAMETER PRO SENSOR ---
SENSOR_CONFIGS = {
    "Ch1":  {"window": "boxcar", "nperseg": 1018, "noverlap": 763}, # 75% von 1018
    "Ch2":  {"window": "boxcar", "nperseg": 608,  "noverlap": 456}, # 75% von 608
    "Wav3": {"window": "hann",   "nperseg": 2048, "noverlap": 1024} # 50% von 2048
}

# Globale Anzeige-Parameter
Y_FREQ_MAX = 200000
X_TIME_MAX = 0.1
SENSORS = ["Ch1", "Ch2", "Wav3"]
R_ID = "00000"

print("Individuelle Sensor-Konfigurationen geladen.")

manager = DataFrameManager(data_dir=data_path)
manager.load_signals()
df = manager.get_dataframe()
print(f"Erfolgreich {len(df)} Signale geladen.")

# Zelle 3: Optimierte Übersicht (STFT)
Visualisierung der Spektrogramme für die verschiedenen Sensoren und Proben (Z01 vs. Z05).

In [ ]:
def plot_optimized_raw(df, r_id):
    fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=True, sharey=True)
    fig.suptitle(f"Zelle 3: Optimierte Übersicht | Aufnahme: {r_id}", fontsize=14)
    for i, s_id in enumerate(SENSORS):
        p = SENSOR_CONFIGS[s_id]
        ov_perc = int((p['noverlap']/p['nperseg'])*100)
        for j, spec in enumerate(['Z01', 'Z05']):
            subset = df[(df['spec'] == spec) & (df['sID'] == s_id) & (df['rID'] == r_id)]
            if subset.empty: continue
            data = subset.iloc[0]
            f, t, Zxx = calculate_stft(data['sig'], data['fs'], **p)
            ax = axes[i, j]
            im = ax.pcolormesh(t, f, np.abs(Zxx), shading='nearest')
            ax.set_ylim(0, Y_FREQ_MAX); ax.set_xlim(0, X_TIME_MAX)
            if j == 0:
                ax.set_ylabel(f"{s_id}\n({p['window']}, {p['nperseg']}, {ov_perc}%)", fontweight='bold', fontsize=9)
            ax.set_title(f"{spec}")
            plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

# Ausführen
plot_optimized_raw(df, R_ID)

# Zelle 4: Differenz-Analyse
Berechnet und visualisiert die Differenz der Amplituden zwischen Z05 und Z01, um Veränderungen hervorzuheben.

In [ ]:
def plot_optimized_difference(df, r_id):
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True, sharey=True)
    fig.suptitle(f"Zelle 4: Differenz-Analyse (Z05 - Z01) | Aufnahme: {r_id}", fontsize=14)
    for i, s_id in enumerate(SENSORS):
        p = SENSOR_CONFIGS[s_id]; ov_perc = int((p['noverlap']/p['nperseg'])*100)
        z01_s = df[(df['spec'] == 'Z01') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        z05_s = df[(df['spec'] == 'Z05') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        if z01_s.empty or z05_s.empty: continue
        f, t, s1 = calculate_stft(z01_s.iloc[0]['sig'], z01_s.iloc[0]['fs'], **p)
        _, _, s5 = calculate_stft(z05_s.iloc[0]['sig'], z05_s.iloc[0]['fs'], **p)
        diff = np.abs(s5) - np.abs(s1)
        ax = axes[i]; lim = np.max(np.abs(diff)) * 0.6
        im = ax.pcolormesh(t, f, diff, shading='nearest', cmap='seismic', vmin=-lim, vmax=lim)
        ax.set_ylim(0, Y_FREQ_MAX); ax.set_xlim(0, X_TIME_MAX)
        ax.set_ylabel(f"{s_id}\n({p['window']}, {p['nperseg']}, {ov_perc}%)", fontweight='bold', fontsize=9)
        plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

# Ausführen
plot_optimized_difference(df, R_ID)

# Zelle 5: Spektrale Analyse (PSD)
Vergleich der durchschnittlichen Leistungsdichte (Power Spectral Density) über die Zeit für Z01 und Z05.

In [ ]:
def plot_optimized_psd(df, r_id):
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
    fig.suptitle(f"Zelle 5: Spektrale Analyse (PSD) | Aufnahme: {r_id}", fontsize=14)
    for i, s_id in enumerate(SENSORS):
        p = SENSOR_CONFIGS[s_id]; ov_perc = int((p['noverlap']/p['nperseg'])*100)
        p01 = df[(df['spec'] == 'Z01') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        p05 = df[(df['spec'] == 'Z05') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        if p01.empty or p05.empty: continue
        f, _, s1 = calculate_stft(p01.iloc[0]['sig'], p01.iloc[0]['fs'], **p)
        _, _, s5 = calculate_stft(p05.iloc[0]['sig'], p05.iloc[0]['fs'], **p)
        psd1, psd5 = np.mean(np.abs(s1), axis=1), np.mean(np.abs(s5), axis=1)
        ax = axes[i]
        ax.plot(f, psd1, 'b--', label="Z01", alpha=0.8); ax.plot(f, psd5, 'r-', label="Z05", alpha=0.6)
        ax.fill_between(f, psd1, psd5, where=(psd1 > psd5), color='blue', alpha=0.1)
        ax.fill_between(f, psd1, psd5, where=(psd5 > psd1), color='red', alpha=0.1)
        ax.set_xlim(0, Y_FREQ_MAX); ax.set_ylabel(f"{s_id}\n({p['window']}, {p['nperseg']}, {ov_perc}%)", fontsize=9)
        ax.legend(loc='upper right', fontsize=8)
    plt.tight_layout(); plt.show()

# Ausführen
plot_optimized_psd(df, R_ID)

# Zelle 6: Dominanz-Analyse (Zeitlich)
Analyse, welches Signal (Z01 oder Z05) über die Zeit hinweg energetisch dominiert.

In [ ]:
def plot_optimized_dominance(df, r_id):
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
    fig.suptitle(f"Zelle 6: Dominanz-Analyse (Zeitlich) | Aufnahme: {r_id}", fontsize=14)
    print(f"--- 📊 Zeitliche Dominanz-Bilanz ---")
    for i, s_id in enumerate(SENSORS):
        p = SENSOR_CONFIGS[s_id]; ov_perc = int((p['noverlap']/p['nperseg'])*100)
        p01 = df[(df['spec'] == 'Z01') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        p05 = df[(df['spec'] == 'Z05') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        if p01.empty or p05.empty: continue
        f, t, s1 = calculate_stft(p01.iloc[0]['sig'], p01.iloc[0]['fs'], **p)
        _, _, s5 = calculate_stft(p05.iloc[0]['sig'], p05.iloc[0]['fs'], **p)
        e1 = np.sum(np.abs(s1[f <= Y_FREQ_MAX, :]), axis=0)
        e5 = np.sum(np.abs(s5[f <= Y_FREQ_MAX, :]), axis=0)
        p1_dom = np.mean(e1 > e5) * 100
        print(f"Sensor {s_id:4}: Z01 dominiert zu {p1_dom:5.1f}%")
        ax = axes[i]
        ax.plot(t, e1, 'b--', alpha=0.5); ax.plot(t, e5, 'r-', alpha=0.5)
        ax.fill_between(t, e1, e5, where=(e1 > e5), color='blue', alpha=0.2)
        ax.fill_between(t, e1, e5, where=(e5 > e1), color='red', alpha=0.2)
        ax.set_ylabel(f"{s_id}\n({ov_perc}%)", fontsize=9); ax.set_xlim(0, X_TIME_MAX)
    plt.tight_layout(); plt.show()

# Ausführen
plot_optimized_dominance(df, R_ID)

# Zelle 7: Top 2 Studie
Automatisierte Suche nach den besten Fenster-Parametern (Klarheit), um die Unterschiede zwischen Z01 und Z05 zu maximieren.

In [ ]:
def run_top2_study(df, r_id):
    nperseg_range = range(128, 2049, 20) # Schrittweite leicht erhöht für Performance
    overlap_ratios = [0.5, 0.75, 0.9]
    window_list = ["hann", "hamming", "blackman", "boxcar"]
    all_configs = []
    print(f"Suche nach den Top 2 für alle Sensoren (Jupyter-Optimiert)...")
    start_t = time.time()
    for s_id in SENSORS:
        p01 = df[(df['spec'] == 'Z01') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        p05 = df[(df['spec'] == 'Z05') & (df['sID'] == s_id) & (df['rID'] == r_id)]
        if p01.empty or p05.empty: continue
        sig1, sig5, fs = p01.iloc[0]['sig'], p05.iloc[0]['sig'], p01.iloc[0]['fs']
        for win in window_list:
            for nps in nperseg_range:
                for ratio in overlap_ratios:
                    nov = int(nps * ratio)
                    f, _, s1 = calculate_stft(sig1, fs, nperseg=nps, noverlap=nov, window=win)
                    _, _, s5 = calculate_stft(sig5, fs, nperseg=nps, noverlap=nov, window=win)
                    e1 = np.sum(np.abs(s1[f <= Y_FREQ_MAX, :]), axis=0)
                    e5 = np.sum(np.abs(s5[f <= Y_FREQ_MAX, :]), axis=0)
                    p1_dom = np.mean(e1 > e5) * 100
                    clarity = abs(p1_dom - 50)
                    all_configs.append({
                        "Sensor": s_id, "Window": win, "NPERSEG": nps, 
                        "Ratio": f"{int(ratio*100)}%", "Dominanz_Z01": round(p1_dom, 1), 
                        "Klarheit": round(clarity, 1)
                    })
    end_t = time.time()
    print(f"Fertig in {end_t - start_t:.1f} Sekunden.")
    res_df = pd.DataFrame(all_configs)
    return res_df.sort_values(['Sensor', 'Klarheit'], ascending=[True, False]).groupby('Sensor').head(2)

# Ausführen und Anzeigen
top2 = run_top2_study(df, R_ID)
display(top2)